In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [40]:
from langchain.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("attention.pdf")
documents = loader.load()

In [42]:
import re

def clean_text(text):
    text = text.replace("\n", " ")   # remove line breaks
    text = re.sub(r'\s+', ' ', text)  # remove extra spaces
    return text

for doc in documents:
    doc.page_content = clean_text(doc.page_content)

In [ ]:
def remove_noise(text):
    text = re.sub(r'\b\d+\b', '', text)  # remove standalone numbers
    return text

for doc in documents:
    doc.page_content = remove_noise(doc.page_content)

In [43]:
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(separator="\n\n",chunk_size=500,chunk_overlap=100)
text=text_splitter.split_documents(documents)

In [44]:
text[0]

Document(page_content='Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗† University of Toronto aidan@cs.toronto.edu Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com Illia Polosukhin∗‡ illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convo

In [45]:
from langchain.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 661.97it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [46]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [47]:
from langchain_community.vectorstores import FAISS
db=FAISS.from_documents(text,embeddings,normalize_L2 = True)
db

In [48]:
Query = "What is attention mechanism?" 
docs1=db.similarity_search(Query)
docs1[0].page_content

'Attention Visualizations It is in this spirit that a majority of American governments have passed new laws since 2009 making the registration or voting process more difficult . <EOS> <pad> <pad> <pad> <pad> <pad> <pad> It is in this spirit that a majority of American governments have passed new laws since 2009 making the registration or voting process more difficult . <EOS> <pad> <pad> <pad> <pad> <pad> <pad> Figure 3: An example of the attention mechanism following long-distance dependencies in the encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for the word ‘making’. Different colors represent different heads. Best viewed in color. 13'

In [49]:
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.7,
        "k": 3
    }
)

In [50]:
def format_docs(text):
    return "\n\n".join(doc.page_content for doc in text)

In [51]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an AI assistant.

Use ONLY the provided context to answer.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
""")

In [52]:
llm = Ollama(
                model="gemma:2b",
                temperature=0.2,
                num_predict=500 
            )

In [53]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [54]:
chain = prompt | llm | parser

In [55]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | parser
)


In [56]:
query = "What is attention mechanism?"

docs = retriever.invoke(query)

print("\n=== RETRIEVED DOCS ===\n")
for doc in docs:
    print(doc.page_content)


=== RETRIEVED DOCS ===



c:\Users\BIT\OneDrive\Desktop\Document Analysis\venv\Lib\site-packages\langchain_core\vectorstores.py:342: UserWarning: No relevant docs were retrieved using the relevance score threshold 0.7
  warnings.warn(


In [59]:
response = rag_chain.invoke("explain self attention in transformers?")
print(response)

c:\Users\BIT\OneDrive\Desktop\Document Analysis\venv\Lib\site-packages\langchain_core\vectorstores.py:342: UserWarning: No relevant docs were retrieved using the relevance score threshold 0.7
  warnings.warn(


Sure, here's an explanation of self attention in transformers:

Self-attention is a mechanism within the transformer architecture that allows each token in the input sequence to attend to all other tokens in the sequence. This mechanism helps to capture long-range dependencies and contextual relationships between different parts of the text.

Self-attention is achieved through the use of attention weights, which are learned during the training process. These attention weights indicate which tokens are most relevant to each other based on their semantic similarity.

Self-attention can be used in various ways, such as:

* **Self-attention within a token:** This type of self-attention focuses on the attention weights between a particular token and all other tokens in the sequence.
* **Self-attention between tokens:** This type of self-attention focuses on the attention weights between all pairs of tokens in the sequence.

Self-attention has been shown to be very effective in various natur

In [ ]:
query = "explain self attention in transformers?"

results = db.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results):
    print(f"\nResult {i+1}:")
    print("Content:", doc.page_content)
    print("Similarity Score:", score)


Result 1:
Content: Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively. 3.1 Encoder and Decoder Stacks Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position- wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding layers, produce outputs of dimension dmodel = 512. Decoder: The decoder is also composed of a stack of N = 

: 